# PoRepMarket — Contract Management

Interactive notebook for all `PoRepMarket` public functions.

| Function | Caller | Description |
|---|---|---|
| `proposeDeal` | client | Create a new deal proposal |
| `acceptDeal` | provider controlling addr | Accept a proposed deal |
| `rejectDeal` | client or provider | Reject a proposed deal |
| `completeDeal` | Client SC only | Mark deal as completed |
| `terminateDeal` | Validator only | Terminate a completed deal |
| `updateManifestLocation` | client | Update the deal manifest URI |
| `setClientSmartContract` | admin | Set Client SC address |
| `grantRole` / `revokeRole` | admin | Manage access control |

Contract addresses and keys are read from `filecoin-boost/scripts/porep-market/.env`.

## 0. Setup

In [ ]:
import builtins as _builtins
import json
import logging
import os
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv
from web3 import Web3
from web3.exceptions import ContractCustomError
from web3.middleware import ExtraDataToPOAMiddleware

# ── .env ──────────────────────────────────────────────────────────────────────
BOOST_ENV = Path.home() / "Forked/filecoin-boost/scripts/porep-market/.env"
if BOOST_ENV.exists():
    load_dotenv(BOOST_ENV, override=True)
    print(f"Loaded .env from {BOOST_ENV}")
else:
    load_dotenv(override=True)
    print(f"WARNING: {BOOST_ENV} not found")

# ── CONFIG ────────────────────────────────────────────────────────────────────
_key = os.getenv("PRIVATE_KEY_TEST", os.getenv("PRIVATE_KEY", ""))
CONFIG = {
    "rpc_url":       os.getenv("RPC_URL", "http://127.0.0.1:1234/rpc/v1"),
    "private_key":   _key,
    "porep_market":  os.getenv("POREP_MARKET", ""),
    "client_sc":     os.getenv("CLIENT_CONTRACT", ""),
    "sp_registry":   os.getenv("SP_REGISTRY", ""),
    "validator_factory": os.getenv("VALIDATOR_FACTORY", ""),
}

_addr_keys = ("porep_market", "client_sc", "sp_registry", "validator_factory")
for _k in _addr_keys:
    if CONFIG[_k]:
        CONFIG[_k] = Web3.to_checksum_address(CONFIG[_k])

if not CONFIG["porep_market"]:
    print("WARNING: POREP_MARKET not set — run 02_deploy.sh first")
else:
    print("Contract address loaded.")

print("\n── CONFIG ──────────────────────────────────────")
for _k, _v in CONFIG.items():
    print(f"  {_k:<25} {_v}")

# ── ABI dir ───────────────────────────────────────────────────────────────────
_nb = globals().get("__vsc_ipynb_file__", "")
ABI_DIR = Path(_nb).parent.parent / "abis" if _nb else BOOST_ENV.parent / "porep-market" / "abis"
assert ABI_DIR.exists(), f"ABI dir not found: {ABI_DIR}"
print(f"\nABI dir : {ABI_DIR}")

def load_abi(name: str):
    with open(ABI_DIR / f"{name}.json") as f:
        return json.load(f)

# ── Logger ────────────────────────────────────────────────────────────────────
_log_dir = Path(_nb).parent / "logs" if _nb else Path.home() / "porep-market-logs"
_log_dir.mkdir(parents=True, exist_ok=True)
if not hasattr(_builtins, "_porep_market_log"):
    _builtins._porep_market_log = _log_dir / f"porep-market-{datetime.now().strftime('%Y%m%d-%H%M%S')}.log"
LOG_FILE = _builtins._porep_market_log

logger = logging.getLogger("porep.market")
logger.setLevel(logging.DEBUG)
if not logger.handlers:
    _fh = logging.FileHandler(LOG_FILE, mode="a")
    _fh.setFormatter(logging.Formatter("%(asctime)s  %(levelname)-5s  %(message)s"))
    _ch = logging.StreamHandler()
    _ch.setFormatter(logging.Formatter("%(message)s"))
    logger.addHandler(_fh)
    logger.addHandler(_ch)
log  = logger.info
logw = logger.warning
loge = logger.error
log(f"=== Session log: {LOG_FILE} ===")

In [ ]:
w3 = Web3(Web3.HTTPProvider(CONFIG["rpc_url"]))
w3.middleware_onion.inject(ExtraDataToPOAMiddleware, layer=0)
assert w3.is_connected(), f"Cannot connect to {CONFIG['rpc_url']}"
chain_id = w3.eth.chain_id
log(f"Connected  chain_id={chain_id}  latest_block={w3.eth.block_number}")

In [ ]:
acct = w3.eth.account.from_key(CONFIG["private_key"]) if CONFIG["private_key"] else None
log(f"Account : {acct.address if acct else 'NOT SET'}")
bal = w3.eth.get_balance(acct.address) if acct else 0
log(f"Balance : {w3.from_wei(bal, 'ether')} FIL")

In [ ]:
porep_market = w3.eth.contract(address=CONFIG["porep_market"], abi=load_abi("PoRepMarket"))
log("PoRepMarket contract loaded.")

# ── Error map ─────────────────────────────────────────────────────────────────
_error_map: dict[str, str] = {}
for _abi_file in ABI_DIR.glob("*.json"):
    try:
        for _entry in json.load(open(_abi_file)):
            if _entry.get("type") == "error":
                _sig = _entry["name"] + "(" + ",".join(i["type"] for i in _entry.get("inputs", [])) + ")"
                _error_map["0x" + w3.keccak(text=_sig).hex()[:8]] = _sig
    except Exception:
        pass

def decode_custom_error(data: str) -> str:
    return _error_map.get(data[:10], f"unknown error {data[:10]}")

def send_tx(fn, account, value=0):
    nonce = w3.eth.get_transaction_count(account.address)
    log(f"  → {fn.fn_name}  from={account.address}  nonce={nonce}")
    try:
        tx = fn.build_transaction({
            "from": account.address,
            "nonce": nonce,
            "gasPrice": w3.eth.gas_price,
            "value": value,
            "chainId": chain_id,
        })
    except ContractCustomError as e:
        err = decode_custom_error(e.data)
        loge(f"  ✗ estimate_gas reverted: {err}")
        raise RuntimeError(f"estimate_gas reverted: {err}") from e
    signed = account.sign_transaction(tx)
    tx_hash = w3.eth.send_raw_transaction(signed.raw_transaction)
    log(f"  ⏳ sent tx={tx_hash.hex()}")
    receipt = w3.eth.wait_for_transaction_receipt(tx_hash, timeout=120)
    if receipt["status"] == 1:
        log(f"  ✅ success  tx={tx_hash.hex()}  gas_used={receipt['gasUsed']}")
    else:
        loge(f"  ❌ reverted  tx={tx_hash.hex()}  gas_used={receipt['gasUsed']}")
        raise RuntimeError(f"Transaction reverted: {tx_hash.hex()}")
    return receipt

DEAL_STATES = {0: "Proposed", 1: "Accepted", 2: "Completed", 3: "Rejected", 4: "Terminated"}

# DealProposal is returned as a tuple:
#   p[0]=dealId  p[1]=client  p[2]=provider  p[3]=requirements  p[4]=terms
#   p[5]=validator  p[6]=state  p[7]=railId  p[8]=manifestLocation
# requirements: (retrievabilityBps, bandwidthMbps, latencyMs, indexingPct)
# terms:        (dealSizeBytes, pricePerSectorPerMonth, durationDays)

def fmt_deal(p):
    log(f"  dealId     : {p[0]}")
    log(f"  client     : {p[1]}")
    log(f"  provider   : {p[2]}")
    log(f"  state      : {DEAL_STATES.get(p[6], p[6])}")
    log(f"  validator  : {p[5]}")
    log(f"  railId     : {p[7]}")
    log(f"  manifest   : {p[8]}")
    t = p[4]
    log(f"  size       : {t[0]:,} bytes")
    log(f"  price/mo   : {t[1]}")
    log(f"  duration   : {t[2]} days")
    r = p[3]
    log(f"  req.retr   : {r[0]} bps")
    log(f"  req.bw     : {r[1]} Mbps")
    log(f"  req.lat    : {r[2]} ms")
    log(f"  req.idx    : {r[3]}%")

log(f"Helpers ready. {len(_error_map)} error selectors loaded.")

---
## 1. Contract constants

In [ ]:
max_duration = porep_market.functions.MAX_DEAL_DURATION_DAYS().call()
admin_role   = porep_market.functions.DEFAULT_ADMIN_ROLE().call()
upgrader_role = porep_market.functions.UPGRADER_ROLE().call()
upgrade_ver   = porep_market.functions.UPGRADE_INTERFACE_VERSION().call()
uuid          = porep_market.functions.proxiableUUID().call()

log(f"MAX_DEAL_DURATION_DAYS    : {max_duration}")
log(f"DEFAULT_ADMIN_ROLE        : {admin_role.hex()}")
log(f"UPGRADER_ROLE             : {upgrader_role.hex()}")
log(f"UPGRADE_INTERFACE_VERSION : {upgrade_ver}")
log(f"proxiableUUID             : {uuid.hex()}")

---
## 2. Roles — inspect

In [ ]:
admin_role   = porep_market.functions.DEFAULT_ADMIN_ROLE().call()
upgrader_role = porep_market.functions.UPGRADER_ROLE().call()

address_to_check = acct.address  # change to any address

for role_name, role_bytes in [("DEFAULT_ADMIN_ROLE", admin_role), ("UPGRADER_ROLE", upgrader_role)]:
    has = porep_market.functions.hasRole(role_bytes, address_to_check).call()
    admin_of = porep_market.functions.getRoleAdmin(role_bytes).call()
    log(f"{role_name}")
    log(f"  {address_to_check} has role : {has}")
    log(f"  role admin               : {admin_of.hex()}")

---
## 3. Query a deal

In [ ]:
DEAL_ID = 1  # ← set deal ID

proposal = porep_market.functions.getDealProposal(DEAL_ID).call()
log(f"getDealProposal({DEAL_ID}):")
fmt_deal(proposal)

In [ ]:
manifest = porep_market.functions.getManifestLocation(DEAL_ID).call()
log(f"getManifestLocation({DEAL_ID}) : {manifest}")

---
## 4. List completed deals

In [ ]:
completed = porep_market.functions.getCompletedDeals().call()
log(f"Completed deals in settlement queue: {len(completed)}")
for p in completed:
    log(f"  dealId={p[0]}  provider={p[2]}  railId={p[7]}  size={p[4][0]:,}")

---
## 5. Propose a deal

Called by the **client** wallet. The registry auto-selects the least-committed matching provider.

**Devnet provider capabilities (04_register_miner.sh):**
| ActorID | Retrievability | Bandwidth | Latency | Indexing | Available |
|---------|---------------|-----------|---------|----------|-----------|
| 1000 | 10000 bps | 1000 Mbps | 100 ms | 100% | 1 GB |
| 1001 | 8000 bps | 500 Mbps | 200 ms | 80% | 5 GB |
| 1002 | 5000 bps | 100 Mbps | 500 ms | 50% | 10 GB |

In [ ]:
assert acct, "PRIVATE_KEY_TEST not set"

requirements = (
    9500,   # retrievabilityBps  — miner 1000 has 10000 ✓
    100,    # bandwidthMbps      — all miners ✓
    0,      # latencyMs          — 0 = no requirement
    0,      # indexingPct        — 0 = no requirement
)
terms = (
    536870912,    # dealSizeBytes     — 512 MiB (fits miner 1000's 1 GB)
    1_000_000,    # pricePerSectorPerMonth
    30,           # durationDays      — must be multiple of 30
)
manifest_location = "ipfs://bafybeigdyrzt5sfp7udm7hu76uh7y26nf3efuylqabf3oclgtqy55fbzdi"

receipt = send_tx(
    porep_market.functions.proposeDeal(requirements, terms, manifest_location),
    acct,
)

ev_logs = porep_market.events.DealProposalCreated().process_receipt(receipt)
assert ev_logs, "DealProposalCreated event not found"
ev = ev_logs[0]["args"]
DEAL_ID = ev["dealId"]
log(f"Deal proposed  dealId={DEAL_ID}  provider={ev['provider']}  totalSize={ev['totalDealSize']:,}")

In [ ]:
# Inspect deal state after proposeDeal
proposal = porep_market.functions.getDealProposal(DEAL_ID).call()
fmt_deal(proposal)
log(f"Auto-accepted: {proposal[6] == 1}")

---
## 6. Accept a deal

The **controlling address of the SP actor** calls `acceptDeal(dealId)`.
Skip if the deal was auto-accepted (state == Accepted already).

In [ ]:
proposal = porep_market.functions.getDealProposal(DEAL_ID).call()
if proposal["state"] == 1:
    log("Deal already Accepted — skipping.")
else:
    receipt = send_tx(porep_market.functions.acceptDeal(DEAL_ID), acct)
    ev_logs = porep_market.events.DealAccepted().process_receipt(receipt)
    if ev_logs:
        ev = ev_logs[0]["args"]
        log(f"Deal accepted  dealId={ev['dealId']}  provider={ev['provider']}")

---
## 7. Reject a deal

Called by the **client** or the **provider controlling address**. Deal must be in `Proposed` state.

In [ ]:
REJECT_DEAL_ID = DEAL_ID  # ← set deal ID to reject

receipt = send_tx(porep_market.functions.rejectDeal(REJECT_DEAL_ID), acct)
ev_logs = porep_market.events.DealRejected().process_receipt(receipt)
if ev_logs:
    ev = ev_logs[0]["args"]
    log(f"Deal rejected  dealId={ev['dealId']}  rejector={ev['rejector']}")

---
## 8. Complete a deal

`completeDeal` is **restricted to the Client SC address** (`NotTheClientSmartContract` if called from elsewhere).
On devnet the deployer controls the Client SC, so `PRIVATE_KEY_TEST` can call it directly.

In [ ]:
COMPLETE_DEAL_ID  = DEAL_ID  # ← set deal ID
ACTUAL_SIZE_BYTES = 536870912  # ← actual data size reported by Client SC

receipt = send_tx(
    porep_market.functions.completeDeal(COMPLETE_DEAL_ID, ACTUAL_SIZE_BYTES),
    acct,
)
ev_logs = porep_market.events.DealCompleted().process_receipt(receipt)
if ev_logs:
    ev = ev_logs[0]["args"]
    log(f"Deal completed  dealId={ev['dealId']}  actualSize={ev['actualSizeBytes']:,}  provider={ev['provider']}")

---
## 9. Terminate a deal

`terminateDeal` is **restricted to the deal's Validator contract**.
On devnet you can call it directly from the deployer key only if the Validator delegates to the caller.

In [ ]:
TERMINATE_DEAL_ID = DEAL_ID           # ← set deal ID
TERMINATOR_ADDR   = acct.address      # ← address recorded as terminator
END_EPOCH         = w3.eth.block_number  # ← epoch at which deal ends

receipt = send_tx(
    porep_market.functions.terminateDeal(TERMINATE_DEAL_ID, TERMINATOR_ADDR, END_EPOCH),
    acct,
)
ev_logs = porep_market.events.DealTerminated().process_receipt(receipt)
if ev_logs:
    ev = ev_logs[0]["args"]
    log(f"Deal terminated  dealId={ev['dealId']}  terminator={ev['terminator']}  endEpoch={ev['endEpoch']}")

---
## 10. Update manifest location

Only the **deal client** can update the manifest URI. Max 2048 bytes.

In [ ]:
UPDATE_DEAL_ID       = DEAL_ID  # ← set deal ID
NEW_MANIFEST_LOCATION = "ipfs://bafybeigdyrzt5sfp7udm7hu76uh7y26nf3efuylqabf3oclgtqy55fbzdi"  # ← new URI

old_manifest = porep_market.functions.getManifestLocation(UPDATE_DEAL_ID).call()
log(f"Current manifest : {old_manifest}")

receipt = send_tx(
    porep_market.functions.updateManifestLocation(UPDATE_DEAL_ID, NEW_MANIFEST_LOCATION),
    acct,
)
ev_logs = porep_market.events.ManifestLocationUpdated().process_receipt(receipt)
if ev_logs:
    ev = ev_logs[0]["args"]
    log(f"Manifest updated  dealId={ev['dealId']}")
    log(f"  old : {ev['oldManifestLocation']}")
    log(f"  new : {ev['newManifestLocation']}")

---
## 11. Admin — set Client SC address

Requires `DEFAULT_ADMIN_ROLE`.

In [ ]:
NEW_CLIENT_SC = CONFIG["client_sc"]  # ← set new Client SC address
assert NEW_CLIENT_SC, "CLIENT_CONTRACT not set in .env"

receipt = send_tx(
    porep_market.functions.setClientSmartContract(NEW_CLIENT_SC),
    acct,
)
ev_logs = porep_market.events.ClientSmartContractUpdated().process_receipt(receipt)
if ev_logs:
    log(f"Client SC updated to {ev_logs[0]['args']['clientSmartContract']}")

---
## 12. Admin — grant / revoke roles

In [ ]:
# Grant or revoke a role on an account
ROLE_TO_GRANT  = porep_market.functions.UPGRADER_ROLE().call()  # ← role bytes32
TARGET_ACCOUNT = acct.address  # ← account to grant/revoke
ACTION         = "grant"       # ← "grant" or "revoke"

before = porep_market.functions.hasRole(ROLE_TO_GRANT, TARGET_ACCOUNT).call()
log(f"hasRole before : {before}")

if ACTION == "grant":
    receipt = send_tx(porep_market.functions.grantRole(ROLE_TO_GRANT, TARGET_ACCOUNT), acct)
else:
    receipt = send_tx(porep_market.functions.revokeRole(ROLE_TO_GRANT, TARGET_ACCOUNT), acct)

after = porep_market.functions.hasRole(ROLE_TO_GRANT, TARGET_ACCOUNT).call()
log(f"hasRole after  : {after}")